# 1. Tìm hiểu bài toán và dữ liệu

Không cần phần trước.

Muốn chạy toàn bộ trong một kernel: [mở notebook 00](00_Gamer_Behavior_Clustering.ipynb).

<details><summary>Danh sách tám phần</summary>

- [1. Tìm hiểu bài toán và dữ liệu](01_data_understanding.ipynb)

- [2. Thống kê và khám phá dữ liệu (EDA)](02_statistics_eda.ipynb)

- [3. Kiểm tra và làm sạch dữ liệu](03_data_cleaning.ipynb)

- [4. Chia dữ liệu và chuẩn hóa](04_data_transformation.ipynb)

- [5. Ward, K-Means và cách kết hợp](05_clustering.ipynb)

- [6. Đánh giá holdout và độ nhạy](06_model_evaluation.ipynb)

- [7. Hồ sơ người chơi và trực quan hóa](07_player_profiling.ipynb)

- [8. Gán người chơi và đề xuất chiến lược](08_prediction_strategy.ipynb)

</details>

## Khởi tạo và đọc dữ liệu

In [1]:
from pathlib import Path
import os, sys, json, hashlib
import numpy as np
import pandas as pd
import joblib
from IPython.display import display, Markdown

# Tìm gốc dự án khi mở từ thư mục notebooks/, gamer_profiling/ hoặc workspace.
cwd = Path.cwd().resolve()
candidates = [cwd, *cwd.parents, cwd / "gamer_profiling"]
ROOT = next((p for p in candidates if (p / "gamer_clustering.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Mở notebook từ thư mục dự án gamer_profiling hoặc notebooks/.")
NBOUT = Path(os.environ.get("GAMER_NOTEBOOK_OUTPUT", str(ROOT / "outputs/notebook/latest")))
NBOUT.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(NBOUT / ".matplotlib"))
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from gamer_clustering import clean_features, centers_from_labels, metrics, choose_k
from sklearn.cluster import KMeans
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import (pairwise_distances, silhouette_score, silhouette_samples,
                             davies_bouldin_score, adjusted_rand_score)
from sklearn.metrics.pairwise import pairwise_distances_argmin
from sklearn.decomposition import PCA
from scipy.cluster.hierarchy import linkage, cut_tree, dendrogram
from threadpoolctl import threadpool_limits
import matplotlib.pyplot as plt
import seaborn as sns

# Module .py dùng Agg khi chạy batch; trong notebook cần hiển thị hình ngay dưới ô code.
get_ipython().run_line_magic("matplotlib", "inline")
thread_limit = threadpool_limits(limits=2)
sns.set_theme(style="whitegrid", font="DejaVu Sans", context="notebook")
pd.set_option("display.max_columns", 20)
pd.set_option("display.precision", 4)
SEED = 42
FEATURES = ["PlayTimeHours", "SessionsPerWeek", "AvgSessionDurationMinutes",
            "PlayerLevel", "AchievementsUnlocked"]
METHODS = ["KMeans", "Ward", "Ward_KMeans"]
SOURCE = ROOT / "data/online_gaming_behavior_dataset.csv"
df = pd.read_csv(SOURCE)
reference = json.loads((ROOT / "outputs/summary.json").read_text(encoding="utf-8"))

def require(name, previous):
    path = NBOUT / name
    if not path.is_file():
        raise FileNotFoundError(f"Chưa có {name}. Chạy notebook {previous} trước, hoặc Run All notebook 00.")
    return path

def save_plot(name):
    plt.tight_layout()
    plt.savefig(NBOUT / name, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close()

Mục tiêu là mô tả những người có hành vi gần nhau bằng **Agglomerative Ward kết hợp K-Means**,
sau đó đánh giá bằng Silhouette và Davies–Bouldin. Đây là học không giám sát.

Nguồn: [Rabie El Kharoua, Kaggle](https://www.kaggle.com/datasets/rabieelkharoua/predict-online-gaming-behavior-dataset),
phiên bản 1, giấy phép **CC BY 4.0**. Tác giả xác nhận đây là **dữ liệu tổng hợp phục vụ giáo dục**.
Chúng ta dùng CSV nguyên bản, không tự tạo người chơi mới.

In [2]:
card = json.loads((ROOT / "data/dataset_card.json").read_text(encoding="utf-8"))
digest = hashlib.sha256(SOURCE.read_bytes()).hexdigest()
assert digest == reference["data_sha256"], "CSV khác bản dùng trong báo cáo."
assert "dataset is synthetic" in card["description"].lower()
display(pd.DataFrame({"Thông tin": ["Số người chơi", "Số cột", "Tác giả", "Giấy phép", "Loại dữ liệu"],
                      "Giá trị": [str(len(df)), str(df.shape[1]), card["creatorName"],
                                  card["licenseName"], "Tổng hợp (synthetic)"]}))
display(df.head())
print("SHA-256:", digest)

,Thông tin,Giá trị
0,Số người chơi,40034
1,Số cột,13
2,Tác giả,Rabie El Kharoua
3,Giấy phép,Attribution 4.0 International (CC BY 4.0)
4,Loại dữ liệu,Tổng hợp (synthetic)


,PlayerID,Age,Gender,Location,GameGenre,PlayTimeHours,InGamePurchases,GameDifficulty,SessionsPerWeek,AvgSessionDurationMinutes,PlayerLevel,AchievementsUnlocked,EngagementLevel
0,9000,43,Male,Other,Strategy,16.2711,0,Medium,6,108,79,25,Medium
1,9001,29,Female,USA,Strategy,5.5260,0,Medium,5,144,11,10,Medium
2,9002,22,Female,USA,Sports,8.2238,0,Easy,16,142,35,41,High
3,9003,35,Male,USA,Action,5.2654,1,Easy,9,85,57,47,Medium
4,9004,33,Male,Europe,Action,15.5319,0,Medium,2,131,95,37,Medium


SHA-256: 63e63057d2984a0630d9af5457df564ac35af5ec0d9d7d82dad6a8e7097f4e15


Một hàng là một `PlayerID`. Chỉ năm biến số được đưa vào khoảng cách; ID bị loại.
`EngagementLevel` và `InGamePurchases` chỉ dùng mô tả sau phân cụm, không dùng chọn k.

In [3]:
roles = {
    "PlayerID": "Khóa truy vết; không dùng học",
    "Age": "Nhân khẩu học; không dùng học", "Gender": "Nhân khẩu học; không dùng học",
    "Location": "Khu vực; không dùng học", "GameGenre": "Ngữ cảnh; không dùng học",
    "PlayTimeHours": "Chỉ báo thời lượng thô; cần xác minh đơn vị",
    "InGamePurchases": "Có mua 0/1; chỉ đối chiếu sau học",
    "GameDifficulty": "Ngữ cảnh; không dùng học",
    "SessionsPerWeek": "Tần suất: số phiên/tuần",
    "AvgSessionDurationMinutes": "Thời lượng: phút/phiên",
    "PlayerLevel": "Tiến trình; không phải điểm kỹ năng",
    "AchievementsUnlocked": "Thành tích tích lũy; không phải tốc độ thu vật phẩm",
    "EngagementLevel": "Nhãn nguồn; chỉ đối chiếu sau học",
}
dictionary = pd.DataFrame({"Cột": df.columns, "Kiểu": df.dtypes.astype(str).values,
                           "Ý nghĩa / vai trò": [roles[c] for c in df.columns]})
display(dictionary)
display(pd.DataFrame([
    ["Tỷ lệ thắng/thua", "Thiếu Wins và Losses"],
    ["Tốc độ thu vật phẩm", "Thiếu số vật phẩm cùng thời gian theo dõi"],
    ["Điểm kỹ năng thật", "Thiếu Elo/MMR hoặc thang kỹ năng"],
    ["High-Spenders", "Thiếu số tiền mua; cờ 0/1 không đo chi tiêu cao"],
], columns=["Chỉ số đề bài gợi ý", "Vì sao chưa tính được"]))

,Cột,Kiểu,Ý nghĩa / vai trò
0,PlayerID,int64,Khóa truy vết; không dùng học
1,Age,int64,Nhân khẩu học; không dùng học
2,Gender,str,Nhân khẩu học; không dùng học
3,Location,str,Khu vực; không dùng học
4,GameGenre,str,Ngữ cảnh; không dùng học
5,PlayTimeHours,float64,Chỉ báo thời lượng thô; cần xác minh đơn vị
6,InGamePurchases,int64,Có mua 0/1; chỉ đối chiếu sau học
7,GameDifficulty,str,Ngữ cảnh; không dùng học
8,SessionsPerWeek,int64,Tần suất: số phiên/tuần
9,AvgSessionDurationMinutes,int64,Thời lượng: phút/phiên


,Chỉ số đề bài gợi ý,Vì sao chưa tính được
0,Tỷ lệ thắng/thua,Thiếu Wins và Losses
1,Tốc độ thu vật phẩm,Thiếu số vật phẩm cùng thời gian theo dõi
2,Điểm kỹ năng thật,Thiếu Elo/MMR hoặc thang kỹ năng
3,High-Spenders,Thiếu số tiền mua; cờ 0/1 không đo chi tiêu cao


**Cách trình bày:** “Nhóm chọn đặc trưng theo những gì nguồn thực sự đo được.
Cấp độ và thành tích mô tả tiến trình; nhóm không đổi tên chúng thành kỹ năng hoặc tốc độ thu vật phẩm.”
Tiếp theo: khám phá phân phối, ngoại lệ và sự nhất quán giữa hai cột thời lượng.